# Parte 2 — experiências de validação, sementes e calibração

Tarefas do plano «parte 2» que precisam de GPU e dos dados no Drive:

| Tarefa | O que faz | Conjunto |
|---|---|---|
| **D2** | M-E0a, M-E0b e M-E0c na mandioca (e resumo da E0 já existente) | validação |
| **D3** | E0, E0a, E0b e E0c do milho com as sementes de treino 1, 2 e 3 + McNemar E0 × E0b | validação |
| **D4** | *Temperature scaling* (T aprendido na validação, aplicado ao teste) e AUROC CCMT × iCassava/CBSD | teste (modelos finais, sem treino) |

Regras: a divisão é a de sempre (`divisao_<cultura>.csv`, semente 42). **O teste do CCMT não é usado
no D2 nem no D3.** O código de dados, modelo e treino é o do `02_treino.ipynb`, sem alterações; só
mudam os parâmetros de cada experiência.

Cada experiência grava os resultados no Drive assim que termina. Se o Colab cair, basta voltar a
correr o notebook: as experiências já feitas são saltadas.

## 1. Versões das bibliotecas (igual ao `02_treino`)

In [ ]:
import importlib.metadata as md
import os
import re
import shutil
import sys

if sys.version_info >= (3, 13):
    raise RuntimeError("O TensorFlow 2.17.1 só suporta até Python 3.12: escolha uma versão anterior "
                       "do ambiente de execução do Colab (ex.: 2025.10), com GPU T4.")

VERSOES = {"tensorflow": "2.17.1", "keras": "3.15.1", "numpy": "1.26.4",
           "scikit-learn": "1.5.2", "pandas": "2.2.3"}
TEM_GPU = shutil.which("nvidia-smi") is not None


def versao_instalada(pacote):
    try:
        return md.version(pacote)
    except md.PackageNotFoundError:
        return None


def bibliotecas_cuda_em_falta():
    em_falta = []
    for requisito in md.requires("tensorflow") or []:
        r = re.match(r"([\w.\-]+)\s*\(?==([^);\s]+)\)?\s*;.*and-cuda", requisito)
        if r and versao_instalada(r.group(1)) != r.group(2):
            em_falta.append(r.group(1))
    return em_falta


em_falta = {p: v for p, v in VERSOES.items() if versao_instalada(p) != v}
if TEM_GPU and not em_falta and bibliotecas_cuda_em_falta():
    em_falta["tensorflow"] = VERSOES["tensorflow"]

if em_falta:
    pacotes = [f"{p}=={v}" for p, v in em_falta.items()]
    if TEM_GPU and "tensorflow" in em_falta:
        pacotes = [f"tensorflow[and-cuda]=={VERSOES['tensorflow']}" if p.startswith("tensorflow==")
                   else p for p in pacotes]
    pacotes = " ".join(f'"{p}"' for p in pacotes)
    print("A instalar:", pacotes)
    !pip install -q {pacotes}
    print("Instalação concluída. A sessão vai reiniciar: corra o notebook outra vez desde o início.")
    os.kill(os.getpid(), 9)
else:
    print("Versões correctas:", VERSOES, "| GPU:", "sim" if TEM_GPU else "não")

## 2. O que correr e onde estão os ficheiros

Confirme os caminhos dos **modelos finais** (D4) e das **imagens de fora do CCMT** (D4.2) no seu Drive.

In [ ]:
TAREFAS = ["D2", "D3", "D4"]   # retire as que não quer correr nesta sessão

SEMENTE_DIVISAO = 42           # NUNCA muda: a divisão vem dos CSV
SEMENTES_TREINO = [1, 2, 3]    # D3
DETERMINISMO = True            # tf.config.experimental.enable_op_determinism(); se der erro
                               # "... does not have a deterministic GPU implementation", pôr False
                               # e reiniciar a sessão (e dizê-lo no relatório)

# Hiperparâmetros comuns (iguais ao 02_treino)
TAMANHO_IMAGEM, TAMANHO_LOTE = 224, 32
EPOCAS_FASE1, EPOCAS_FASE2, PACIENCIA = 20, 30, 5
LR_FASE1, DROPOUT = 1e-3, 0.3

# Configurações (fase 2). "ultimas_20" = configuração base E0.
BLOCOS = ("block6", "block7", "top")
CONFIGURACOES = {
    "E0":  dict(descongelar="ultimas_20", lr_fase2=1e-5, label_smoothing=0.0),
    "E0a": dict(descongelar=BLOCOS,       lr_fase2=1e-4, label_smoothing=0.0),
    "E0b": dict(descongelar=BLOCOS,       lr_fase2=5e-5, label_smoothing=0.0),
    "E0c": dict(descongelar=BLOCOS,       lr_fase2=5e-5, label_smoothing=0.1),
}

# Pasta da experiência E0 da mandioca que já existe (para a linha "E0" do D2).
PASTA_E0_MANDIOCA = "resultados/mandioca/ccmt"

# D4: modelos finais (os que deram 87,0 % e 93,1 % no teste). Ajuste se estiverem noutra pasta.
MODELOS_FINAIS = {
    "milho":    "resultados/milho/E0b/modelo_milho.keras",
    "mandioca": "resultados/mandioca/ccmt/modelo_mandioca.keras",
}
# D4.2: imagens de fora do CCMT (positivos = "fora do âmbito"). Pastas no Drive, relativas a PFC/.
FORA_DO_AMBITO = {
    "icassava": "externo/icassava_80",   # as 80 imagens do teste externo (A1)
    "cbsd":     "externo/cbsd_10",       # as 10 imagens de estria castanha
}

## 3. Bibliotecas, Drive e dados

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus[0].name if gpus else "NENHUMA — Ambiente de execução → GPU T4")
if DETERMINISMO:
    tf.config.experimental.enable_op_determinism()

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PFC = Path("/content/drive/MyDrive/PFC")
    PASTA_RAW = Path("/content/raw")
    if not PASTA_RAW.exists():
        print("A descomprimir ccmt_raw.zip (alguns minutos)...")
        shutil.unpack_archive(str(PASTA_PFC / "ccmt_raw.zip"), str(PASTA_RAW))
    PASTA_DIVISOES = PASTA_PFC / "divisoes"
else:
    PASTA_PFC = Path.cwd().parent / "models"
    PASTA_RAW = Path.cwd().parent / "data" / "raw"
    PASTA_DIVISOES = Path.cwd().parent / "data" / "divisoes"

PASTA_P2 = PASTA_PFC / "resultados" / "parte2"
PASTA_P2.mkdir(parents=True, exist_ok=True)


def ler_divisao(cultura):
    d = pd.read_csv(PASTA_DIVISOES / f"divisao_{cultura}.csv")
    classes = sorted(d["classe"].unique())
    d["rotulo"] = d["classe"].map({c: i for i, c in enumerate(classes)})
    d["ficheiro"] = [str(PASTA_RAW / c) for c in d["caminho"]]
    em_falta = [f for f in d["ficheiro"] if not Path(f).exists()]
    assert not em_falta, f"{len(em_falta)} imagens em falta, ex.: {em_falta[:3]}"
    return d, classes


for c in ("milho", "mandioca"):
    d, _ = ler_divisao(c)
    print(c, d["subconjunto"].value_counts().to_dict())

## 4. Dados, modelo e treino — o código do `02_treino`, em funções

As únicas diferenças são de organização: o código está dentro de funções para poder correr várias
experiências seguidas, e a semente do treino (`semente`) passa a ser um parâmetro. Ela fixa os pesos
da cabeça, a ordem dos lotes e o aumento de dados; a divisão não muda.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def criar_aumento():
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(30 / 360),
        layers.RandomZoom(0.2),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
        layers.RandomContrast(0.2),
    ], name="aumento_de_dados")


def criar_dataset(df, n_classes, treino, semente, aumento=None):
    def ler_imagem(caminho, rotulo):
        imagem = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
        imagem = tf.image.resize(imagem, (TAMANHO_IMAGEM, TAMANHO_IMAGEM))
        imagem = tf.cast(tf.clip_by_value(tf.round(imagem), 0, 255), tf.uint8)
        return imagem, tf.one_hot(rotulo, n_classes)

    ds = tf.data.Dataset.from_tensor_slices((df["ficheiro"].values, df["rotulo"].values))
    ds = ds.map(ler_imagem, num_parallel_calls=AUTOTUNE).cache()
    if treino:
        ds = ds.shuffle(len(df), seed=semente, reshuffle_each_iteration=True)
    ds = ds.batch(TAMANHO_LOTE)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if treino:
        ds = ds.map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def treinar(cultura, nome, descongelar, lr_fase2, label_smoothing, semente):
    """Treina uma configuração e grava em PASTA_P2/<cultura>/<nome>/. Devolve a pasta."""
    saida = PASTA_P2 / cultura / nome
    if (saida / "resumo.json").exists():
        print(f"[{cultura}/{nome}] já feita — a saltar")
        return saida
    saida.mkdir(parents=True, exist_ok=True)
    keras.utils.set_random_seed(semente)

    d, classes = ler_divisao(cultura)
    n = len(classes)
    treino_df = d[(d["subconjunto"] == "treino") & (d["fonte"] == "ccmt")]
    val_df = d[d["subconjunto"] == "validacao"]
    aumento = criar_aumento()
    treino_ds = criar_dataset(treino_df, n, True, semente, aumento)
    val_ds = criar_dataset(val_df, n, False, semente)
    pesos = dict(enumerate(compute_class_weight("balanced", classes=np.arange(n), y=treino_df["rotulo"])))

    base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                             input_shape=(TAMANHO_IMAGEM, TAMANHO_IMAGEM, 3))
    base.trainable = False
    entradas = keras.Input(shape=(TAMANHO_IMAGEM, TAMANHO_IMAGEM, 3), name="imagem")
    x = base(entradas, training=False)
    x = layers.GlobalAveragePooling2D(name="pooling")(x)
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    saidas = layers.Dense(n, activation="softmax", name="classificacao")(x)
    modelo = keras.Model(entradas, saidas, name=f"efficientnetb0_{cultura}")

    def callbacks(fase):
        return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA,
                                              restore_best_weights=True, verbose=1),
                keras.callbacks.CSVLogger(str(saida / f"historial_{fase}.csv"))]

    perda = keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)
    modelo.compile(optimizer=keras.optimizers.Adam(LR_FASE1), loss=perda, metrics=["accuracy"])
    t0 = time.time()
    h1 = modelo.fit(treino_ds, validation_data=val_ds, epochs=EPOCAS_FASE1,
                    class_weight=pesos, callbacks=callbacks("fase1"), verbose=2)
    t1 = time.time()

    base.trainable = True
    ultimas_20 = set(c.name for c in base.layers[-20:])
    for camada in base.layers:
        treinavel = (camada.name in ultimas_20 if descongelar == "ultimas_20"
                     else camada.name.startswith(tuple(descongelar)))
        camada.trainable = treinavel and not isinstance(camada, layers.BatchNormalization)
    n_treinaveis = sum(c.trainable for c in base.layers)
    modelo.compile(optimizer=keras.optimizers.Adam(lr_fase2), loss=perda, metrics=["accuracy"])
    feitas = len(h1.epoch)
    h2 = modelo.fit(treino_ds, validation_data=val_ds, initial_epoch=feitas,
                    epochs=feitas + EPOCAS_FASE2, class_weight=pesos,
                    callbacks=callbacks("fase2"), verbose=2)
    t2 = time.time()

    # Avaliação final na VALIDAÇÃO (o teste não é tocado).
    perda_val, acc_val = modelo.evaluate(val_ds, verbose=0)
    prob = modelo.predict(val_ds, verbose=0)
    np.save(saida / "validacao_probabilidades.npy", prob)
    rel = pd.DataFrame(classification_report(val_df["rotulo"].values, prob.argmax(1),
                                             labels=range(n), target_names=classes,
                                             output_dict=True, zero_division=0)).T
    rel.to_csv(saida / "validacao_metricas_por_classe.csv")

    def melhor(h):  # época com menor val_loss (a que o EarlyStopping repõe)
        i = int(np.argmin(h.history["val_loss"]))
        return i + 1, float(h.history["val_accuracy"][i])

    (m1, acc1), (m2, acc2) = melhor(h1), melhor(h2)
    resumo = {
        "cultura": cultura, "experiencia": nome, "semente_treino": semente,
        "semente_divisao": SEMENTE_DIVISAO, "descongelar": descongelar if isinstance(descongelar, str)
        else list(descongelar), "camadas_treinaveis_base": n_treinaveis, "lr_fase2": lr_fase2,
        "label_smoothing": label_smoothing, "determinismo": DETERMINISMO,
        "fase1": {"epocas": len(h1.epoch), "melhor_epoca": m1, "val_acc": round(acc1, 4)},
        "fase2": {"epocas": len(h2.epoch), "melhor_epoca": m2, "val_acc": round(acc2, 4)},
        "val_acc_final": round(float(acc_val), 4), "val_loss_final": round(float(perda_val), 4),
        "val_f1_ponderado": round(float(rel.loc["weighted avg", "f1-score"]), 4),
        "tempo_min": round((t2 - t0) / 60, 1), "tempo_fase1_min": round((t1 - t0) / 60, 1),
        "imagens": {"treino": len(treino_df), "validacao": len(val_df)},
        "gpu": gpus[0].name if gpus else None,
    }
    (saida / "resumo.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(resumo, ensure_ascii=False))
    keras.backend.clear_session()
    return saida

## 5. D2 — variações na mandioca (validação)

M-E0a, M-E0b e M-E0c com a semente 42 (como a E0). A linha E0 lê os ficheiros que o `02_treino`
já deixou em `PASTA_E0_MANDIOCA`.

In [ ]:
def linha_resumo(pasta):
    """Linha da tabela D2 a partir dos ficheiros de uma experiência (nova ou do 02_treino)."""
    pasta = Path(pasta)
    rel = pd.read_csv(pasta / "validacao_metricas_por_classe.csv", index_col=0)
    if (pasta / "resumo.json").exists():
        r = json.loads((pasta / "resumo.json").read_text(encoding="utf-8"))
        f1, f2 = r["fase1"], r["fase2"]
        linha = {"F1 épocas": f1["epocas"], "F1 val_acc": f1["val_acc"], "F2 épocas": f2["epocas"],
                 "F2 val_acc": f2["val_acc"], "val_loss": r["val_loss_final"],
                 "val_acc final": r["val_acc_final"], "tempo (min)": r["tempo_min"]}
    else:  # experiência do 02_treino
        reg = json.loads(next(pasta.glob("registo_treino_*.json")).read_text(encoding="utf-8"))
        linha = {}
        for fase, rotulo in (("fase1", "F1"), ("fase2", "F2")):
            h = pd.read_csv(pasta / f"historial_{fase}.csv")
            linha[f"{rotulo} épocas"] = len(h)
            linha[f"{rotulo} val_acc"] = round(float(h.loc[h["val_loss"].idxmin(), "val_accuracy"]), 4)
        linha.update({"val_loss": reg["validacao"]["perda"], "val_acc final": reg["validacao"]["acuracia"],
                      "tempo (min)": round(sum(reg["duracao_min"].values()), 1)})
    linha["F1 ponderado"] = round(float(rel.loc["weighted avg", "f1-score"]), 4)
    return linha, rel


if "D2" in TAREFAS:
    for nome in ("E0a", "E0b", "E0c"):
        treinar("mandioca", f"M-{nome}", semente=SEMENTE_DIVISAO, **CONFIGURACOES[nome])

    pastas = {"E0 (existente)": PASTA_PFC / PASTA_E0_MANDIOCA}
    pastas.update({f"M-{n}": PASTA_P2 / "mandioca" / f"M-{n}" for n in ("E0a", "E0b", "E0c")})
    linhas, f1_classes = {}, {}
    for nome, pasta in pastas.items():
        linhas[nome], rel = linha_resumo(pasta)
        _, classes = ler_divisao("mandioca")
        f1_classes[nome] = rel.loc[classes + ["weighted avg"], "f1-score"].round(3)
    D2_TABELA = pd.DataFrame(linhas).T
    D2_F1 = pd.DataFrame(f1_classes)
    D2_TABELA.to_csv(PASTA_P2 / "D2_mandioca_tabela.csv")
    D2_F1.to_csv(PASTA_P2 / "D2_mandioca_f1_por_classe.csv")
    display(D2_TABELA)
    display(D2_F1)

## 6. D3 — milho: 4 configurações × 3 sementes de treino (validação)

12 treinos (cerca de 5 h numa T4). Pode correr de noite; se o Colab cair, volte a correr: o que já
está feito é saltado.

In [ ]:
if "D3" in TAREFAS:
    for semente in SEMENTES_TREINO:
        for nome, cfg in CONFIGURACOES.items():
            treinar("milho", f"{nome}_s{semente}", semente=semente, **cfg)

In [ ]:
from scipy.stats import binomtest, chi2

if "D3" in TAREFAS:
    linhas = []
    for nome in CONFIGURACOES:
        for semente in SEMENTES_TREINO:
            p = PASTA_P2 / "milho" / f"{nome}_s{semente}" / "resumo.json"
            if p.exists():
                r = json.loads(p.read_text(encoding="utf-8"))
                linhas.append({"config": nome, "semente": semente, "val_acc": r["val_acc_final"],
                               "val_loss": r["val_loss_final"], "F1 ponderado": r["val_f1_ponderado"],
                               "épocas": r["fase1"]["epocas"] + r["fase2"]["epocas"],
                               "tempo (min)": r["tempo_min"]})
    D3_TABELA = pd.DataFrame(linhas)
    D3_RESUMO = D3_TABELA.groupby("config", sort=False).agg(
        n=("val_acc", "size"), val_acc_media=("val_acc", "mean"), val_acc_dp=("val_acc", "std"),
        f1_media=("F1 ponderado", "mean"), f1_dp=("F1 ponderado", "std")).round(4)
    D3_TABELA.to_csv(PASTA_P2 / "D3_milho_sementes.csv", index=False)
    D3_RESUMO.to_csv(PASTA_P2 / "D3_milho_resumo.csv")
    display(D3_TABELA)
    display(D3_RESUMO)

    # As diferenças entre configurações são maiores do que a variação entre sementes?
    amplitude_configs = D3_RESUMO["val_acc_media"].max() - D3_RESUMO["val_acc_media"].min()
    dp_medio = D3_RESUMO["val_acc_dp"].mean()
    print(f"Amplitude entre médias das configurações: {amplitude_configs:.4f} | "
          f"DP médio entre sementes: {dp_medio:.4f}")

    # McNemar E0 × E0b (semente 1), mesmas imagens de validação.
    d, classes = ler_divisao("milho")
    y = d[d["subconjunto"] == "validacao"]["rotulo"].values
    a = np.load(PASTA_P2 / "milho" / "E0_s1" / "validacao_probabilidades.npy").argmax(1) == y
    b = np.load(PASTA_P2 / "milho" / "E0b_s1" / "validacao_probabilidades.npy").argmax(1) == y
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())  # só E0 acerta / só E0b acerta
    exacto = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    qui2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10) if n01 + n10 else 0.0
    D3_MCNEMAR = {"n": int(len(y)), "acertos_E0": int(a.sum()), "acertos_E0b": int(b.sum()),
                  "so_E0_acerta": n01, "so_E0b_acerta": n10, "p_exacto": round(float(exacto), 4),
                  "qui2_corrigido": round(float(qui2), 3), "p_qui2": round(float(chi2.sf(qui2, 1)), 4)}
    print("McNemar E0 × E0b (semente 1):", D3_MCNEMAR)

## 7. D4 — calibração (temperature scaling) e imagens de fora do CCMT

Usa os **modelos finais** (sem treino). Os *logits* calculam-se exactamente: saída da camada
`pooling` × pesos da camada `classificacao` (sem o softmax). T aprende-se só na **validação** e
aplica-se depois ao **teste**. O modelo não muda, por isso isto não é um uso novo do teste.

In [ ]:
from PIL import Image, ImageOps
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax, softmax


def extrator_logits(caminho_modelo):
    m = keras.models.load_model(caminho_modelo, compile=False)
    feats = keras.Model(m.inputs, m.get_layer("pooling").output)
    W, b = m.get_layer("classificacao").get_weights()
    return lambda ds: feats.predict(ds, verbose=0) @ W + b


def ds_imagens(ficheiros):
    def ler(c):
        im = tf.io.decode_image(tf.io.read_file(c), channels=3, expand_animations=False)
        im = tf.image.resize(im, (TAMANHO_IMAGEM, TAMANHO_IMAGEM))
        return tf.clip_by_value(tf.round(im), 0, 255)
    return (tf.data.Dataset.from_tensor_slices(list(ficheiros)).map(ler, num_parallel_calls=AUTOTUNE)
            .batch(TAMANHO_LOTE).prefetch(AUTOTUNE))


def nll(z, y, T=1.0):
    return float(-log_softmax(z / T, axis=1)[np.arange(len(y)), y].mean())


def ece(p, y, intervalos=15):
    conf, prev = p.max(1), p.argmax(1)
    limites = np.linspace(0, 1, intervalos + 1)
    total = 0.0
    for lo, hi in zip(limites[:-1], limites[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((prev[m] == y[m]).mean() - conf[m].mean())
    return float(total)


def brier(p, y):
    return float(((p - np.eye(p.shape[1])[y]) ** 2).sum(1).mean())


def risco_cobertura(p, y, limiares=(0.0, 0.6, 0.8, 0.9)):
    conf, certo = p.max(1), p.argmax(1) == y
    erros = int((~certo).sum())
    linhas = []
    for t in limiares:
        cobre = conf >= t
        linhas.append({"τ": t, "cobertura (%)": round(100 * cobre.mean(), 1),
                       "sem aviso": int(cobre.sum()),
                       "exactidão sem aviso (%)": round(100 * certo[cobre].mean(), 2) if cobre.any() else None,
                       "erros avisados": f"{int((~certo & ~cobre).sum())} / {erros}",
                       "fotos com aviso (%)": round(100 * (~cobre).mean(), 1)})
    return pd.DataFrame(linhas)


if "D4" in TAREFAS:
    D4 = {}
    for cultura, rel_modelo in MODELOS_FINAIS.items():
        d, classes = ler_divisao(cultura)
        logits = extrator_logits(PASTA_PFC / rel_modelo)
        zv = logits(ds_imagens(d[d["subconjunto"] == "validacao"]["ficheiro"]))
        yv = d[d["subconjunto"] == "validacao"]["rotulo"].values
        zt = logits(ds_imagens(d[d["subconjunto"] == "teste"]["ficheiro"]))
        yt = d[d["subconjunto"] == "teste"]["rotulo"].values
        T = float(minimize_scalar(lambda t: nll(zv, yv, t), bounds=(0.05, 10), method="bounded").x)
        p0, pT = softmax(zt, 1), softmax(zt / T, 1)
        metricas = pd.DataFrame({
            "antes (T=1)": {"ECE": ece(p0, yt), "Brier": brier(p0, yt), "NLL": nll(zt, yt)},
            f"depois (T={T:.3f})": {"ECE": ece(pT, yt), "Brier": brier(pT, yt), "NLL": nll(zt, yt, T)},
        }).round(4)
        rc0, rcT = risco_cobertura(p0, yt), risco_cobertura(pT, yt)
        D4[cultura] = {"T": T, "n_val": len(yv), "n_teste": len(yt),
                       "acertos_teste": int((p0.argmax(1) == yt).sum()), "zt": zt}
        np.save(PASTA_P2 / f"D4_{cultura}_logits_teste.npy", zt)
        metricas.to_csv(PASTA_P2 / f"D4_{cultura}_calibracao.csv")
        rc0.to_csv(PASTA_P2 / f"D4_{cultura}_risco_cobertura_T1.csv", index=False)
        rcT.to_csv(PASTA_P2 / f"D4_{cultura}_risco_cobertura_T.csv", index=False)
        print(f"\n=== {cultura}: T = {T:.3f} | teste n = {len(yt)}, acertos = {D4[cultura]['acertos_teste']}")
        display(metricas)
        print("Risco–cobertura sem T:"); display(rc0)
        print(f"Risco–cobertura com T = {T:.3f}:"); display(rcT)

In [ ]:
if "D4" in TAREFAS:
    # D4.2: a confiança separa o teste do CCMT (negativos) das imagens de outra origem (positivos)?
    extensoes = {".jpg", ".jpeg", ".png"}
    fora = {nome: sorted(p for p in (PASTA_PFC / pasta).rglob("*") if p.suffix.lower() in extensoes)
            for nome, pasta in FORA_DO_AMBITO.items()}
    print({k: len(v) for k, v in fora.items()})
    logits = extrator_logits(PASTA_PFC / MODELOS_FINAIS["mandioca"])
    z_fora = logits(ds_imagens([str(p) for v in fora.values() for p in v]))
    z_ccmt = D4["mandioca"]["zt"]
    T = D4["mandioca"]["T"]
    linhas = []
    for rotulo, t in (("sem T", 1.0), (f"com T = {T:.3f}", T)):
        c_fora, c_ccmt = softmax(z_fora / t, 1).max(1), softmax(z_ccmt / t, 1).max(1)
        y = np.r_[np.ones(len(c_fora)), np.zeros(len(c_ccmt))]
        pont = 1 - np.r_[c_fora, c_ccmt]
        n_ic = len(fora["icassava"])
        linhas.append({"pontuação": rotulo, "AUROC": round(roc_auc_score(y, pont), 4),
                       "n fora": len(c_fora), "n CCMT": len(c_ccmt),
                       "iCassava com aviso (<0,60)": f"{int((c_fora[:n_ic] < 0.6).sum())} / {n_ic}",
                       "CBSD com aviso (<0,60)": f"{int((c_fora[n_ic:] < 0.6).sum())} / {len(c_fora) - n_ic}",
                       "CCMT com aviso (<0,60)": f"{int((c_ccmt < 0.6).sum())} / {len(c_ccmt)}"})
    D4_OOD = pd.DataFrame(linhas)
    D4_OOD.to_csv(PASTA_P2 / "D4_fora_do_ambito.csv", index=False)
    display(D4_OOD)

## 8. Relatório (Markdown) para colar no relatório da parte 2

In [ ]:
partes = [f"# Resultados do notebook 03_parte2 — {time.strftime('%Y-%m-%d %H:%M')}",
          f"GPU: {gpus[0].name if gpus else 'nenhuma'} | determinismo: {DETERMINISMO}"]
if "D2" in TAREFAS:
    partes += ["## D2. Mandioca, validação", D2_TABELA.to_markdown(), "F1 por classe:", D2_F1.to_markdown()]
if "D3" in TAREFAS:
    partes += ["## D3. Milho, sementes", D3_TABELA.to_markdown(index=False), "Médias ± DP:",
               D3_RESUMO.to_markdown(), f"McNemar E0 × E0b (semente 1): {D3_MCNEMAR}"]
if "D4" in TAREFAS:
    partes.append("## D4. Calibração e fora do âmbito")
    for cultura in MODELOS_FINAIS:
        partes += [f"### {cultura}: T = {D4[cultura]['T']:.3f}",
                   pd.read_csv(PASTA_P2 / f"D4_{cultura}_calibracao.csv", index_col=0).to_markdown(),
                   "Sem T:", pd.read_csv(PASTA_P2 / f"D4_{cultura}_risco_cobertura_T1.csv").to_markdown(index=False),
                   "Com T:", pd.read_csv(PASTA_P2 / f"D4_{cultura}_risco_cobertura_T.csv").to_markdown(index=False)]
    partes.append(D4_OOD.to_markdown(index=False))
texto = "\n\n".join(partes)
(PASTA_P2 / "relatorio_parte2_colab.md").write_text(texto, encoding="utf-8")
print(texto)